In [4]:
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import pandas as pd
import os 
# Configuration constants
MIN_POSTS_PER_USER = 2

# Plan
1. Consider just the profiles that are present in chunk_0_posts_parquet: "user of interests". From them create a dictionary for fast joining_date_lookups.
2. "posts" time filtering of two weeks. 
3. We can further reduce the users of interest by asking if it has at least two posts. 
3. "blocks", "follows", "likes" filtering based on:
   1. The event happened in the first week after the joining date. 
   2. The user is in the user of interests. 

NB: We should create our final dataset consider the posts in chunks.

# User of Interests
We're going to study the users present in "chunk_0_posts" which have a bluesky joining date.

Furthermore we're filtering out users with $\le 1$ posts. 

In [5]:
# Load the posts data
posts_path = "../data/posting/cleaned/chunk_0_posts.parquet"
posts_table = pq.read_table(posts_path)

posts_df = posts_table.to_pandas()

print(f"Initial posts count: {len(posts_df)}")
print(f"Unique users: {posts_df['did_id'].nunique()}")

user_post_counts = posts_df.groupby('did_id').size().reset_index(name='post_count')

# Filter out users with less than MIN_POSTS_PER_USER posts.
active_posters = user_post_counts[user_post_counts['post_count'] >= MIN_POSTS_PER_USER]['did_id'].values
filtered_posts_df = posts_df[posts_df['did_id'].isin(active_posters)]

print(f"\nPosts after filtering users with <{MIN_POSTS_PER_USER} posts: {len(filtered_posts_df)}")
print(f"Active posters (≥{MIN_POSTS_PER_USER} posts): {len(active_posters)}")

Initial posts count: 4994663
Unique users: 130111

Posts after filtering users with <2 posts: 4957527
Active posters (≥2 posts): 92975

Posts after filtering users with <2 posts: 4957527
Active posters (≥2 posts): 92975


In [6]:
# Load filter profiles for users of interest only
profiles_path = "../data/posting/cleaned/profiles.parquet"
profiles_table = pq.read_table(profiles_path)

# Filter to active_posters only
profiles_table = profiles_table.filter(pc.is_in(profiles_table['did_id'], pa.array(active_posters)))
user_of_interests = profiles_table.to_pandas()

print(f"Users of interests (active posters -- ≥{MIN_POSTS_PER_USER} posts -- with a join date): {len(user_of_interests)}")

Users of interests (active posters -- ≥2 posts -- with a join date): 65380


In [7]:
# Create a fast lookup dictionary: did_id -> join_date
join_date_dict = dict(zip(
    user_of_interests['did_id'],
    user_of_interests['created_at']
))

print(f"Join date lookup dictionary created: {len(join_date_dict)} entries")

Join date lookup dictionary created: 65376 entries


# Event Filtering
We filter an event (e.g. block, follow) if:
1. It doesn't involve an user of interest.
2. It happens after one week since the joining date.

In [25]:
# Generic function to filter event databases with multithreading (vectorized with PyArrow)
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np

# Pre-create a set and PyArrow array of active user IDs for faster filtering
active_user_ids_set = set(join_date_dict.keys())
active_user_ids_array = pa.array(list(active_user_ids_set))

# Pre-compute max_days_in_ns for vectorized comparison
MAX_DAYS_IN_NS = np.timedelta64(1, 'D').astype('timedelta64[ns]').astype('int64')

def filter_events(input_path, output_path, db_name, max_days, user_type="two_user", num_threads=4):
    """
    Filter events to only include those within max_days of user joining.
    
    Args:
        input_path: Path to input parquet file
        output_path: Path to output parquet file
        db_name: Name of the database being filtered (for logging)
        max_days: Maximum days since user joining to keep event
        user_type: "single_user" for posts (only did_id) or "two_user" for blocks/follows/likes (did_id and subject_id)
        num_threads: Number of threads for parallel processing
    
    For single_user events (posts): keeps if days_since_user_join <= max_days
    For two_user events (blocks, follows, likes): keeps if event within max_days of either user's join
    """
    print(f"Filtering {db_name} by row groups (using {num_threads} threads, vectorized)...")
    print(f"Max days filter: {max_days} days | Event type: {user_type}")
    
    pf = pq.ParquetFile(input_path)
    print(f"Total {db_name}: {pf.metadata.num_rows:,}")
    print(f"Row groups: {pf.num_row_groups}")
    
    input_size = os.path.getsize(input_path)
    print(f"Input file size: {input_size / (1024**3):.2f} GB")
    
    filtered_tables = {}
    total_rows_processed = 0
    total_rows_kept = 0
    
    max_days_ns = max_days * MAX_DAYS_IN_NS
    
    def process_row_group(rg_index):
        """Process a single row group using vectorized operations"""
        row_group_table = pf.read_row_group(rg_index)
        rows_in_group = row_group_table.num_rows
        
        # Step 1: Use PyArrow for filtering user of intersts
        if user_type == "single_user":
            # For single-user events, only check did_id
            if 'did_id' in row_group_table.column_names:
                mask = pc.is_in(row_group_table['did_id'], active_user_ids_array)
                filtered_table = row_group_table.filter(mask)
            else:
                filtered_table = row_group_table
        else:  # user_type == "two_user"
            # For two-user events, check either did_id or subject_id
            mask = None
            
            if 'did_id' in row_group_table.column_names:
                did_mask = pc.is_in(row_group_table['did_id'], active_user_ids_array)
                mask = did_mask
            
            if 'subject_id' in row_group_table.column_names:
                subject_mask = pc.is_in(row_group_table['subject_id'], active_user_ids_array)
                if mask is None:
                    mask = subject_mask
                else:
                    mask = pc.or_(mask, subject_mask)
            
            filtered_table = row_group_table.filter(mask) if mask is not None else row_group_table
        
        # Step 2: Apply time-based filtering using vectorized operations
        if filtered_table.num_rows > 0 and max_days is not None:
            # Convert to pandas only for time filtering lookup
            filtered_df = filtered_table.to_pandas()
            
            # Vectorized approach: map join dates to all rows
            if user_type == "single_user":
                # Single-user: get join date for each did_id
                join_dates = filtered_df['did_id'].map(join_date_dict)
                days_since_join = (filtered_df['created_at'] - join_dates).dt.days
                time_mask = (days_since_join >= 0) & (days_since_join <= max_days)
            else:  # user_type == "two_user"
                # Two-user: check both did_id and subject_id
                join_dates_did = filtered_df['did_id'].map(join_date_dict)
                days_since_join_did = (filtered_df['created_at'] - join_dates_did).dt.days
                time_mask_did = (days_since_join_did >= 0) & (days_since_join_did <= max_days)
                
                join_dates_subject = filtered_df['subject_id'].map(join_date_dict)
                days_since_join_subject = (filtered_df['created_at'] - join_dates_subject).dt.days
                time_mask_subject = (days_since_join_subject >= 0) & (days_since_join_subject <= max_days)
                
                # Keep row if either user's event is within max_days
                time_mask = time_mask_did | time_mask_subject
            
            filtered_df = filtered_df[time_mask]
            filtered_table = pa.Table.from_pandas(filtered_df, preserve_index=False)
        
        return rg_index, filtered_table, filtered_table.num_rows, rows_in_group
    
    # Use ThreadPoolExecutor to process row groups in parallel
    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = {executor.submit(process_row_group, i): i for i in range(pf.num_row_groups)}
        
        for future in as_completed(futures):
            rg_index, filtered_table, rows_kept, rows_total = future.result()
            filtered_tables[rg_index] = filtered_table
            total_rows_processed += rows_total
            total_rows_kept += rows_kept
            
            print(f"✅ Row group {rg_index+1}/{pf.num_row_groups}: {rows_total:,} → {rows_kept:,} rows")
    
    # Write all filtered tables in order
    print("\nWriting filtered results...")
    writer = None
    for i in range(pf.num_row_groups):
        filtered_table = filtered_tables[i]
        
        # Initialize writer with first filtered table schema
        if writer is None:
            writer = pq.ParquetWriter(
                output_path,
                filtered_table.schema,
                compression='zstd',
                use_dictionary=True,
                write_statistics=True
            )
        
        # Write filtered row group
        if filtered_table.num_rows > 0:
            writer.write_table(filtered_table)
    
    if writer:
        writer.close()
    
    # Calculate statistics
    retention_rate = (total_rows_kept / total_rows_processed * 100) if total_rows_processed > 0 else 0
    output_size = os.path.getsize(output_path)
    size_reduction = input_size - output_size
    size_reduction_percent = (size_reduction / input_size * 100) if input_size > 0 else 0
    
    print("\n" + "="*60)
    print(f"{db_name.upper()} FILTERING SUMMARY:")
    print("="*60)
    print(f"Row groups processed: {pf.num_row_groups} (with {num_threads} threads)")
    print(f"Rows:       {total_rows_processed:,} → {total_rows_kept:,} ({retention_rate:.1f}% kept)")
    print(f"File size:  {input_size / (1024**3):.2f} GB → {output_size / (1024**3):.2f} GB")
    print(f"Reduction:  {size_reduction / (1024**3):.2f} GB ({size_reduction_percent:.1f}%)")
    print(f"Output: {output_path}")
    print("="*60)

In [26]:
# Filter chunk_0_posts
# Posts are single-user events filtered to 14 days after joining
posts_input_path = "../data/posting/cleaned/chunk_0_posts.parquet"
posts_output_path = "../data/posting/filtered/chunk_0_posts.parquet"

filter_events(posts_input_path, posts_output_path, "posts", max_days=14, user_type="single_user", num_threads=1)

Filtering posts by row groups (using 1 threads, vectorized)...
Max days filter: 14 days | Event type: single_user
Total posts: 4,994,663
Row groups: 5
Input file size: 0.03 GB
✅ Row group 1/5: 1,047,600 → 98,119 rows
✅ Row group 2/5: 1,048,110 → 104,551 rows
✅ Row group 3/5: 1,047,810 → 104,028 rows
✅ Row group 2/5: 1,048,110 → 104,551 rows
✅ Row group 3/5: 1,047,810 → 104,028 rows
✅ Row group 4/5: 1,047,134 → 102,208 rows
✅ Row group 5/5: 804,009 → 83,596 rows

Writing filtered results...
✅ Row group 4/5: 1,047,134 → 102,208 rows
✅ Row group 5/5: 804,009 → 83,596 rows

Writing filtered results...

POSTS FILTERING SUMMARY:
Row groups processed: 5 (with 1 threads)
Rows:       4,994,663 → 492,502 (9.9% kept)
File size:  0.03 GB → 0.00 GB
Reduction:  0.03 GB (87.0%)
Output: ../data/posting/filtered/chunk_0_posts.parquet

POSTS FILTERING SUMMARY:
Row groups processed: 5 (with 1 threads)
Rows:       4,994,663 → 492,502 (9.9% kept)
File size:  0.03 GB → 0.00 GB
Reduction:  0.03 GB (87.0%)
Ou

In [ ]:
# Filter blocks
# Blocks are two-user events filtered to 7 days after either user's joining
blocks_input_path = "../data/posting/cleaned/blocks.parquet"
blocks_output_path = "../data/posting/filtered/blocks.parquet"

filter_events(blocks_input_path, blocks_output_path, "blocks", max_days=7, user_type="two_user")

Filtering blocks by row groups (using 4 threads)...
Total blocks: 120,084,926
Row groups: 978
Input file size: 1.45 GB
✅ Row group 4/978: 122,880 → 269 rows
✅ Row group 2/978: 122,878 → 394 rows
✅ Row group 4/978: 122,880 → 269 rows
✅ Row group 2/978: 122,878 → 394 rows
✅ Row group 1/978: 122,880 → 839 rows
✅ Row group 1/978: 122,880 → 839 rows
✅ Row group 3/978: 122,880 → 194 rows
✅ Row group 3/978: 122,880 → 194 rows
✅ Row group 7/978: 122,880 → 146 rows
✅ Row group 7/978: 122,880 → 146 rows
✅ Row group 8/978: 122,880 → 347 rows
✅ Row group 5/978: 122,880 → 150 rows
✅ Row group 8/978: 122,880 → 347 rows
✅ Row group 5/978: 122,880 → 150 rows
✅ Row group 6/978: 122,880 → 172 rows
✅ Row group 6/978: 122,880 → 172 rows
✅ Row group 9/978: 122,880 → 273 rows
✅ Row group 9/978: 122,880 → 273 rows
✅ Row group 12/978: 122,880 → 193 rows
✅ Row group 12/978: 122,880 → 193 rows
✅ Row group 11/978: 122,880 → 203 rows
✅ Row group 11/978: 122,880 → 203 rows
✅ Row group 10/978: 122,880 → 228 rows
✅ 

In [ ]:
# Filter follows

In [ ]:
# Filter likes